In [1]:
"""
将单倍型频率和表型信息合并，并且做成可以直接用于机器学习的训练集和测试集
CatBoost可以读入缺失值的，其他的深度学习不可以，所以要另外构建数据集
"""
import os
import numpy as np
import pandas as pd

from sklearn import preprocessing

WORK_PATH = "../processData/9-0-runOnline/"

In [2]:
df_matrix = pd.read_table(os.path.join(WORK_PATH, "phen_matrix.txt"))

df_per = pd.read_table(os.path.join(WORK_PATH, "6000sample_150gene_hap_per.txt"))
df_per = df_per[df_per["sample"].isin(df_matrix["ID"].unique())].copy()

df_merge = df_matrix.merge(df_per, left_on="ID", right_on="sample", how="left")
df_merge = df_merge.drop(columns=["sample"])

In [3]:
df_merge

,ID,location,year,1000-grain_weight,Coleoptile_length,Grain_length,Grain_length_to_width_ratio,Grain_per_panicle,Grain_width,Heading_date,...,LOC_Os11g45600,LOC_Os11g45620,LOC_Os11g37570,LOC_Os12g10540,LOC_Os11g37620,LOC_Os12g18360,LOC_Os12g18729,LOC_Os12g28750,LOC_Os12g34850,LOC_Os12g43550
0,D1,Beijing,2022.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.073082,0.030423,0.285218,0.453373,0.707507,0.255787,0.175430,0.476687,0.077216,0.397983
1,D1,Beijing,2023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.073082,0.030423,0.285218,0.453373,0.707507,0.255787,0.175430,0.476687,0.077216,0.397983
2,D1,Donggang,2022.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.073082,0.030423,0.285218,0.453373,0.707507,0.255787,0.175430,0.476687,0.077216,0.397983
3,D1,Donggang,2023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.073082,0.030423,0.285218,0.453373,0.707507,0.255787,0.175430,0.476687,0.077216,0.397983
4,D1,Gaoan,2022.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.073082,0.030423,0.285218,0.453373,0.707507,0.255787,0.175430,0.476687,0.077216,0.397983
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119105,D999,Yinchuan,2022.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.065642,0.000165,0.605655,0.453373,0.074405,0.348710,0.015708,0.476687,0.267857,0.235780
119106,D999,Yinchuan,2023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.065642,0.000165,0.605655,0.453373,0.074405,0.348710,0.015708,0.476687,0.267857,0.235780
119107,D999,Yinchuan,2022.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.065642,0.000165,0.605655,0.453373,0.074405,0.348710,0.015708,0.476687,0.267857,0.235780
119108,D999,Yinchuan,2023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.065642,0.000165,0.605655,0.453373,0.074405,0.348710,0.015708,0.476687,0.267857,0.235780


In [4]:
#for item in ['1000-grain_weight', 'Yield_per_plant', 'Heading_date','Plant_height','Grain_length_to_width_ratio']:
for item in ['Tiller_number', 'Grain_per_panicle', 'Seed_set_rate']:
    tmp_data = df_merge.copy().dropna(subset=item)
    # 保留 NaN比例 <=50% 的列
    tmp_data = tmp_data.loc[:, tmp_data.isna().mean() <= 0.5].copy()
    
    tmp_data = tmp_data.rename(columns={x:"fea_{}".format(x) for x in tmp_data.columns if x not in [item, 'ID', 'location', 'year']})
    tmp_data.to_csv(os.path.join(WORK_PATH, "trainTestData/data-{}.txt".format(item)), sep="\t", index=0)

    # 每个 location + year 抽样 10%
    test_df = (
        tmp_data.groupby(["location", "year"], group_keys=False)
        .apply(lambda x: x.sample(frac=0.1, random_state=42))
    )
    # 剩余为训练集
    train_df = tmp_data.drop(test_df.index).copy()

    feat_col = [c for c in tmp_data.columns if c.startswith("fea_")]
    target_col = item

    scale_tool = preprocessing.StandardScaler()
    scale_tool.fit(train_df.loc[:, feat_col])
    train_df.loc[:, feat_col] = scale_tool.transform(train_df.loc[:, feat_col])
    test_df.loc[:, feat_col] = scale_tool.transform(test_df.loc[:, feat_col])

    train_df.to_csv(os.path.join(WORK_PATH, "trainTestData/train-{}.txt".format(item)), sep="\t", index=0)
    test_df.to_csv(os.path.join(WORK_PATH, "trainTestData/test-{}.txt".format(item)), sep="\t", index=0)
    
print("ALL DONE")

/tmp/ipykernel_2057053/962356710.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_2057053/962356710.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_2057053/962356710.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior

ALL DONE


In [3]:
#for item in ['1000-grain_weight','Yield_per_plant','Heading_date','Plant_height','Grain_length_to_width_ratio']:
for item in ['Tiller_number', 'Grain_per_panicle', 'Seed_set_rate']:

    tmp_data = df_merge.copy().dropna(subset=[item])

    # 保留 NaN比例 <=50% 的列
    tmp_data = tmp_data.loc[:, tmp_data.isna().mean() <= 0.5].copy()

    tmp_data = tmp_data.rename(
        columns={x: f"fea_{x}" for x in tmp_data.columns if x not in [item,'ID','location','year']}
    )

    feature_cols = [c for c in tmp_data.columns if c.startswith("fea_")]

    # =========================
    # Step1 先划分 train/test
    # =========================

    test_df = (
        tmp_data.groupby(["location","year"], group_keys=False)
        .apply(lambda x: x.sample(frac=0.1, random_state=42))
    )

    train_df = tmp_data.drop(test_df.index).copy()

    # =========================
    # Step2 用 train 计算 median
    # =========================

    train_group_median = (
        train_df.groupby(["location","year"])[feature_cols]
        .median()
    )

    global_median = train_df[feature_cols].median()

    # =========================
    # Step3 定义 fill 函数
    # =========================

    def fill_by_group(df, group_median, global_median):

        df = df.copy()

        for (loc, year), idx in df.groupby(["location","year"]).groups.items():

            if (loc,year) in group_median.index:

                med = group_median.loc[(loc,year)]

                df.loc[idx, feature_cols] = df.loc[idx, feature_cols].fillna(med)

        df[feature_cols] = df[feature_cols].fillna(global_median)

        return df

    # =========================
    # Step4 填补缺失
    # =========================

    train_df = fill_by_group(train_df, train_group_median, global_median)
    test_df  = fill_by_group(test_df,  train_group_median, global_median)

    # =========================
    # Step5 标准化（只用train）
    # =========================

    scaler = preprocessing.StandardScaler()

    scaler.fit(train_df[feature_cols])

    train_df[feature_cols] = scaler.transform(train_df[feature_cols])
    test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

    # =========================
    # 保存数据
    # =========================

    train_df.to_csv(
        os.path.join(WORK_PATH,f"trainTestData/trainFill-{item}.txt"),
        sep="\t",
        index=False
    )

    test_df.to_csv(
        os.path.join(WORK_PATH,f"trainTestData/testFill-{item}.txt"),
        sep="\t",
        index=False
    )

print("ALL DONE")

/tmp/ipykernel_2062188/3840782391.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_2062188/3840782391.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_2062188/3840782391.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behav

ALL DONE


In [ ]:
df_matrix = pd.read_table(os.path.join(WORK_PATH, "phen_matrix.txt"))

with open("../../inbred_3605_name", "w") as f:
    for item in df_matrix["ID"].unique():
        print(item, file=f)